# 🔬 Notebook 2 — Statistical Analysis
**Goal:** Validate assumptions needed for time series modelling.

**Contents:**
1. Setup
2. STL Decomposition (Volume + Value)
3. Stationarity Tests — ADF & KPSS
4. Differencing Visualisation
5. ACF / PACF Plots
6. Feature Correlation Heatmap


## 1. Setup

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
warnings.filterwarnings("ignore")
os.makedirs("plots", exist_ok=True)

DATA_PATH = "data/UPI_Master_2021_2025.csv"
plt.rcParams.update({
    "figure.facecolor":"#FAFAFA","axes.facecolor":"#FAFAFA",
    "axes.grid":True,"grid.alpha":0.3,"font.family":"DejaVu Sans",
    "axes.spines.top":False,"axes.spines.right":False,
})
BLUE,RED,ORANGE,PURPLE = "#1A6FBF","#D62728","#E07B39","#9467BD"

df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
df["Festival_Name"] = df["Festival_Name"].fillna("")
df["Month_Num"] = df["Date"].dt.month
df["Quarter"]   = df["Date"].dt.quarter
series_vol = df.set_index("Date")["Volume (In Mn.)"]
series_val = df.set_index("Date")["Value (In Cr.)"]
print("Data loaded:", df.shape)


## 2. STL Decomposition — Volume (period = 7 days)

In [ ]:
stl = STL(series_vol, period=7, robust=True).fit()
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
for ax, data, label, color in zip(axes,
    [series_vol, stl.trend, stl.seasonal, stl.resid],
    ["Observed","Trend","Seasonal","Residual"],
    [BLUE, RED, ORANGE, "gray"]):
    ax.plot(data.index, data.values, color=color, lw=0.9)
    ax.set_ylabel(label, fontsize=10)
    if label=="Residual": ax.axhline(0,color="black",lw=0.8,linestyle="--")
axes[0].set_title("STL Decomposition — Daily UPI Volume (Weekly Seasonality)",
                  fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/11_stl_decomposition_volume.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# STL for Value
stl_val = STL(series_val, period=7, robust=True).fit()
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
for ax, data, label, color in zip(axes,
    [series_val, stl_val.trend, stl_val.seasonal, stl_val.resid],
    ["Observed","Trend","Seasonal","Residual"],
    [ORANGE, RED, BLUE, "gray"]):
    ax.plot(data.index, data.values, color=color, lw=0.9)
    ax.set_ylabel(label, fontsize=10)
    if label=="Residual": ax.axhline(0,color="black",lw=0.8,linestyle="--")
axes[0].set_title("STL Decomposition — Daily UPI Value",fontsize=13,fontweight="bold")
plt.tight_layout()
plt.savefig("plots/11b_stl_decomposition_value.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Stationarity Tests — ADF & KPSS

In [ ]:
def run_adf(series, name):
    r = adfuller(series.dropna(), autolag="AIC")
    verdict = "✅ STATIONARY" if r[1]<0.05 else "❌ NON-STATIONARY"
    print(f"\nADF Test — {name}")
    print(f"  Statistic : {r[0]:.4f}  |  p-value: {r[1]:.6f}  →  {verdict}")
    print(f"  Critical  : 1%={r[4]['1%']:.3f}  5%={r[4]['5%']:.3f}  10%={r[4]['10%']:.3f}")
    return r[1]

def run_kpss(series, name):
    r = kpss(series.dropna(), regression="c", nlags="auto")
    verdict = "✅ STATIONARY" if r[1]>0.05 else "❌ NON-STATIONARY"
    print(f"\nKPSS Test — {name}")
    print(f"  Statistic : {r[0]:.4f}  |  p-value: {r[1]:.6f}  →  {verdict}")

print("="*55, "\nRAW SERIES\n"+"="*55)
run_adf(series_vol, "Volume (Raw)")
run_kpss(series_vol, "Volume (Raw)")
run_adf(series_val, "Value (Raw)")

print("\n"+"="*55, "\n1ST DIFFERENCED\n"+"="*55)
run_adf(series_vol.diff().dropna(), "Volume (1st diff)")
run_kpss(series_vol.diff().dropna(), "Volume (1st diff)")

print("\n"+"="*55, "\nSEASONAL DIFF (lag=7)\n"+"="*55)
run_adf(series_vol.diff(7).dropna(), "Volume (seasonal diff 7)")


## 4. Differencing Visualisation

In [ ]:
diff1  = series_vol.diff().dropna()
sdiff7 = series_vol.diff(7).dropna()

fig, axes = plt.subplots(3, 1, figsize=(16, 10))
for ax, s, title_, color in zip(axes,
    [series_vol, diff1, sdiff7],
    ["Original Volume","1st Difference","Seasonal Difference (lag=7)"],
    [BLUE, ORANGE, RED]):
    ax.plot(s.index, s.values, color=color, lw=0.7)
    ax.set_title(title_, fontweight="bold")
    if title_ != "Original Volume":
        ax.axhline(0, color="black", lw=0.8, linestyle="--")

plt.tight_layout()
plt.savefig("plots/12_differencing.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. ACF / PACF Plots
> Use these to determine ARIMA (p, d, q) parameters manually.
> - **ACF** cuts off at lag q → MA order
> - **PACF** cuts off at lag p → AR order

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
plot_acf (series_vol.dropna(), ax=axes[0][0], lags=50, color=BLUE,   title="ACF — Volume (Raw)")
plot_pacf(series_vol.dropna(), ax=axes[0][1], lags=50, color=BLUE,   title="PACF — Volume (Raw)", method="ywm")
plot_acf (diff1.dropna(),      ax=axes[1][0], lags=50, color=ORANGE, title="ACF — Volume (1st Diff)")
plot_pacf(diff1.dropna(),      ax=axes[1][1], lags=50, color=ORANGE, title="PACF — Volume (1st Diff)", method="ywm")
for ax in axes.flat:
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.suptitle("ACF / PACF — UPI Volume", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("plots/13_acf_pacf_volume.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Feature Correlation Heatmap

In [ ]:
feat_cols = ["Volume (In Mn.)","Value (In Cr.)","Is_Weekend","Is_Weekday",
             "Is_Festival","Is_Holiday_Adjacent","Is_Long_Weekend",
             "Holiday_Cluster_7D","Day_Number","Month_Num","Quarter"]
corr = df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/14_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
